In [6]:
import polars as pl
import pandera.polars as pa
from datetime import datetime
from src.ecommerce_etl import config
from src.ecommerce_etl.io import read_csv_raw, write_parquet
from src.ecommerce_etl.transforms import (
    parse_datetime, strip_strings, lowercase, uppercase,
    drop_null_keys, deduplicate_by_key,
)

Customers

In [20]:
customers = read_csv_raw("customers")
s_customers = customers.sample(fraction=0.1, seed=42)
s_customers.head(5)

customer_id,customer_name,email,city,state,created_at
str,str,str,str,str,str
"""CUST-00001""","""Beatriz Pereira""","""beatriz.pereira_1@email.com""","""Salvador""","""BA""","""2022-05-14 01:01:00"""
"""CUST-00003""","""Diego Costa""","""diego.costa_3@email.com""","""Campinas""","""SP""","""2022-01-05 11:38:00"""
"""CUST-00004""","""Patricia Cardoso""","""patricia.cardoso_4@email.com""","""Belem""","""PA""","""2022-01-10 18:12:00"""
"""CUST-00008""","""Larissa Araujo""","""larissa.araujo_8@email.com""","""Caruaru""","""PE""","""2022-05-27 18:25:00"""
"""CUST-00006""","""Beatriz Oliveira""","""beatriz.oliveira_6@email.com""","""Goiania""","""GO""","""2023-08-03 22:35:00"""


In [13]:
class Customers(pa.DataFrameModel):
    customer_id: str = pa.Field(nullable=False, unique=True)
    customer_name: str = pa.Field(nullable=True)
    email: str = pa.Field(nullable=True)
    city: str = pa.Field(nullable=True)
    state: str = pa.Field(nullable=True)
    created_at: datetime = pa.Field(nullable=False)
    source_file: str = pa.Field(alias="_source_file", nullable=False)
    ingested_at: datetime = pa.Field(alias="_ingested_at", nullable=False)

    class Config:
        coerce = False
        strict = True

In [14]:
customers_bronze = s_customers.with_columns(
    pl.lit("customers.csv").alias("_source_file"),
    pl.lit(datetime.now()).alias("_ingested_at")
)
write_parquet(customers_bronze, config.BRONZE_DIR / "customers.parquet")

In [15]:
STRING_COLS = ["customer_id", "customer_name", "email", "city", "state"]

customers = parse_datetime(customers_bronze, "created_at")
customers = strip_strings(customers, STRING_COLS)
customers = lowercase(customers, "email")
customers = uppercase(customers, "state")

customers, rej_null = drop_null_keys(customers, "customer_id")
customers, rej_dup = deduplicate_by_key(customers, "customer_id", "created_at")

customers_silver = customers
customers_rejected = pl.concat([
    rej_null.with_columns(pl.lit("null_customer_id").alias("reject_reason")),
    rej_dup.with_columns(pl.lit("duplicate_customer_id").alias("reject_reason")),
], how="diagonal")

write_parquet(customers_silver, config.SILVER_DIR / "customers.parquet")
write_parquet(customers_rejected, config.QUARANTINE_DIR / "customers.parquet")

In [ ]:
quality_rows = []
def record_check(check_name, table, column, checked, failed, stage):
    quality_rows.append({
        "check_name": check_name, "table": table, "column": column,
        "records_checked": checked, "records_failed": failed,
        "pct_failed": round(100 * failed / checked, 2) if checked else 0.0,
        "stage": stage, "executed_at": datetime.now(),
    })

for col in ["customer_id", "customer_name", "email", "city", "state"]:
    record_check("null_check", "customers", col, customers_bronze.height,
                 customers_bronze[col].null_count(), "bronze")
record_check("duplicate_check", "customers", "customer_id", customers_bronze.height,
             customers_bronze.height - customers_bronze["customer_id"].n_unique(), "bronze")

record_check("null_check", "customers", "customer_id", customers_silver.height,
             customers_silver["customer_id"].null_count(), "silver")
record_check("duplicate_check", "customers", "customer_id", customers_silver.height,
             customers_silver.height - customers_silver["customer_id"].n_unique(), "silver")

quality = pl.DataFrame(quality_rows)
write_parquet(quality, config.QUALITY_DIR / "customers.parquet")
quality

In [ ]:
try:
    Customers.validate(customers_silver, lazy=True)
    print("Customers silver table passed")
except pa.errors.SchemaErrors as e:
    display(e.failure_cases)

Contrato Customers: OK ✅
